In [1]:
import os

In [2]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MLops\\DataScienceProject_1\\Health_premium_calculator'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name : str
    params_grid : dict
    target_column: str   

In [6]:
from src.datascience.constants import *
from src.datascience.utils.common import *

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self,var:str) -> ModelTrainerConfig:

        config = self.config.model_trainer
        params = self.params.models
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            params_grid = params[var],
            target_column = schema.name
            
        )
        return model_trainer_config





In [9]:
import pandas as pd
import os
from src.datascience import logger
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
import joblib
from sklearn.model_selection import GridSearchCV


In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self,var:str,accuracy:float):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column.lower()], axis=1)
        test_x = test_data.drop([self.config.target_column.lower()], axis=1)
        train_y = train_data[[self.config.target_column.lower()]]
        test_y = test_data[[self.config.target_column.lower()]]

        if var == 'ElasticNet' :
            model = ElasticNet() 
            clf = GridSearchCV(model, self.config.params_grid,cv=5, scoring='r2',return_train_score=False)
            clf.fit(train_x, train_y.values.ravel()) 
            curr_accuracy = clf.score(test_x, test_y.values.ravel()) 
            if(curr_accuracy>accuracy) :
                accuracy = curr_accuracy 
                joblib.dump(clf, os.path.join(self.config.root_dir, self.config.model_name))

        else :
            model = RandomForestRegressor()
            clf = GridSearchCV(model, self.config.params_grid, cv=5, scoring='r2', return_train_score=False)
            clf.fit(train_x, train_y.values.ravel())
            curr_accuracy = clf.score(test_x, test_y.values.ravel()) 
            if(curr_accuracy>accuracy) :
                accuracy = curr_accuracy 
                joblib.dump(clf, os.path.join(self.config.root_dir, self.config.model_name))

        return accuracy 








In [11]:
model_names = ["ElasticNet", "RandomForestRegressor"]

accuracy = float('-inf')
for model_name in model_names:
    try:
        config = ConfigurationManager()
        model_trainer_config = config.get_model_trainer_config(model_name)
        model_trainer = ModelTrainer(config=model_trainer_config)
        accuracy = model_trainer.train(model_name,accuracy)
    except Exception as e:
        raise e
    

[2025-05-01 22:06:26,783:INFO:common:yaml file: config\config.yaml loaded successfully]
[2025-05-01 22:06:26,790:INFO:common:yaml file: params.yaml loaded successfully]
[2025-05-01 22:06:26,796:INFO:common:yaml file: schema.yaml loaded successfully]
artifacts already exists, skipping.
[2025-05-01 22:06:29,320:INFO:common:yaml file: config\config.yaml loaded successfully]
[2025-05-01 22:06:29,325:INFO:common:yaml file: params.yaml loaded successfully]
[2025-05-01 22:06:29,328:INFO:common:yaml file: schema.yaml loaded successfully]
artifacts already exists, skipping.
artifacts/model_trainer already exists, skipping.
